## Day 2 — Linear Regression

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


### task1: Train a LinearRegression model on a provided regression dataset (e.g. house prices).

In [15]:
df = pd.read_csv('Housing Prices/housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [16]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [17]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())
df.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64

The median was used to fill the missing values in `total_bedrooms` instead 
of the mean, because the median is not affected by outliers, unlike the 
mean, which gets pulled toward them and would give a value that doesn't 
truly represent the data.

In [18]:
df.drop('ocean_proximity', axis=1, inplace=True)
y = df['median_house_value']
X = df.drop('median_house_value', axis=1)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [20]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully")
print("Train R²:", model.score(X_train, y_train))
print("Test R²:", model.score(X_test, y_test))

Model trained successfully
Train R²: 0.6400947924305294
Test R²: 0.6138664756435179


### task2: Report the model's coefficients and identify which feature has the strongest effect.

In [21]:
# Get feature names and their coefficients
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_
})

coefficients = coefficients.sort_values(by='Coefficient', key=abs, ascending=False)
print(coefficients)

print(f"\nIntercept: {model.intercept_}")

# Strongest effect
strongest = coefficients.iloc[0]
print(f"\nStrongest feature: {strongest['Feature']} (coefficient: {strongest['Coefficient']:.2f})")

              Feature   Coefficient
0           longitude -42632.391717
1            latitude -42450.071863
7       median_income  40538.404387
2  housing_median_age   1182.809649
4      total_bedrooms    116.260128
6          households     46.342572
5          population    -38.492213
3         total_rooms     -8.187977

Intercept: -3578224.2348177778

Strongest feature: longitude (coefficient: -42632.39)


The coefficients show how each feature affects the predicted target, assuming the other features remain constant.

- A **positive coefficient** means that increasing the feature is associated with an increase in the predicted target.
- A **negative coefficient** means that increasing the feature is associated with a decrease in the predicted target.
- The **magnitude** (absolute value) of the coefficient indicates the strength of the effect, although coefficients should ideally be compared when features are on similar scales.

The model has an intercept of approximately **-3,578,224.23**.

The strongest coefficient by absolute magnitude is **longitude**, with a coefficient of approximately **-42,632.39**. This means that, according to the fitted model, an increase of one unit in longitude is associated with a decrease of about **$42,632** in the predicted target, while holding the other features constant.

However, longitude should not automatically be considered the most important feature because the features are measured on different scales. Standardizing the features would allow a more meaningful comparison of coefficient magnitudes.

> Note: Coefficient magnitudes should be interpreted carefully because the features have different scales.

### task3: Evaluate the model with MAE, RMSE, and R² on the test set.

In [23]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 51810.48362804246
RMSE: 71133.17349286022
R²: 0.6138664756435179


The model was evaluated on the test set using three metrics:

- **MAE:** 51,810.48 — the model's predictions differ from the actual values by about 51,810 on average.
- **RMSE:** 71,133.17 — the larger value compared with MAE indicates that some predictions have relatively large errors.
- **R²:** 0.614 — the model explains approximately **61.4% of the variation** in the target variable.

Overall, the model shows a moderate predictive performance, but there is still room for improvement.

### task4: Compare the RMSE against a baseline that predicts the mean for every row, and state whether the model adds value.

In [24]:
# Baseline: predict the training mean for every test sample
baseline_pred = np.full(len(y_test), y_train.mean())

# Calculate baseline RMSE
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("Model RMSE:", rmse)
print("Baseline RMSE:", baseline_rmse)

if rmse < baseline_rmse:
    print("The model adds value compared with the baseline.")
else:
    print("The model does not improve over the baseline.")

Model RMSE: 71133.17349286022
Baseline RMSE: 114485.63543099792
The model adds value compared with the baseline.


The baseline model predicts the mean target value for every test sample.

- **Model RMSE:** 71,133.17
- **Baseline RMSE:** 114,485.64

The model has a substantially lower RMSE than the baseline, indicating that it makes more accurate predictions than simply predicting the mean for every sample.

Therefore, **the model adds predictive value** compared with the baseline.